In [1]:
%%time

# Import libraries
import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score,
    KFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# Load dataset
df_model = pd.read_csv(
    'transformed_daily_data/daily_data_5y_30k_features1.csv'
)

# Prepare dataset

# Ensure datetime formatting
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# Sort dataset
df_model = df_model.sort_values(
    ['ticker', 'timestamp']
).copy()


# Next day closing price
df_model['next_close'] = (
    df_model.groupby('ticker')['close'].shift(-1)
)

# Remove final row per ticker for which next_close does not exist
df_model = df_model[
    df_model['next_close'].notna()
].copy()


# Next close change
df_model['next_close_change'] = df_model['next_close'] - df_model['close']


# Next close change PCT - This will be the target
df_model['next_close_change_pct'] = (
    (df_model['next_close'] - df_model['close']) / df_model['close']
) * 100



# Target already created with dataset loading


# Remove final row per ticker for which next_close does not exist)
df_model = df_model[
    df_model['next_close'].notna()
].copy()



# Train/test split

# Train on 2021-2024 data
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

# Test on 2025 data
test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()


# Select features - exclude index, target, and a couple columns that will leak too much
exclude_cols = [
    'timestamp',
    'ticker',
    'next_close',
    'next_close_change',
    'next_close_change_pct'
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]


X_train = train_df[feature_cols]
y_train = train_df['next_close_change_pct']

X_test = test_df[feature_cols]
y_test = test_df['next_close_change_pct']


# Define model
model = xgb.XGBRegressor(random_state=42)



# Add GridSearch (not doing it in initial notebook, just saving my place)


# Train model
model.fit(X_train, y_train)


# Get predictinos
y_test_pred = model.predict(X_test)


# Define evaluation metrics
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

# Print evaluation metrics
print("\n===== REGRESSION METRICS =====")

print(f"MAE:    {mae:.4f}")
print(f"MSE:    {mse:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MedAE:  {medae:.4f}")
print(f"R²:     {r2:.4f}")


===== REGRESSION METRICS =====
MAE:    1.9086
MSE:    10.6581
RMSE:   3.2647
MedAE:  1.1877
R²:     -0.0323
CPU times: user 6.02 s, sys: 1.42 s, total: 7.44 s
Wall time: 3.84 s


In [ ]:
results_df = test_df.copy()

# Add predictions
results_df['pred_next_close_change_pct'] = (y_test_pred).round(2)

# Impute error
results_df['pred_error'] = results_df['pred_next_close_change_pct'] - results_df['next_close_change_pct']

# Absolute error
results_df['abs_pred_error'] = results_df['pred_error'].abs()

# Directionally correct
results_df['directionally_correct'] = (
    np.sign(results_df['next_close_change_pct']) ==
    np.sign(results_df['pred_next_close_change_pct'])
).astype(int)

results_df

In [ ]:
results_df['directionally_correct'].value_counts()

In [ ]:
# Save
results_df.to_csv('models_daily_results/Round_1_RFR.csv', index=False)